In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import HTML, display
from src.to_html import cell, dropdown, heading, paragraph, table

In [2]:
data_dir = Path("../monthly_processed_data")
files = sorted(data_dir.glob("data-*.parquet"))[-2:]

df_list = []
for file in files:
    df_list.append(pd.read_parquet(file)[[
    "delay_in_min", "station_name", "train_number", "train_type",
    "arrival_is_canceled", "departure_is_canceled",
    "is_replacement_train", "replaced_train_number",
    ]])
df = pd.concat(df_list, ignore_index=True)

# Train numbers whose scheduled service was provided by a replacement train
# (t="e"). Those stops are not counted as cancelled because the service still
# ran, just under a different train number.
replaced_numbers = set(df.loc[df["is_replacement_train"], "replaced_train_number"].dropna())
df["has_replacement"] = df["train_number"].isin(replaced_numbers)
# ran = the stop was actually served (includes additional stops and replacement trains)
df["ran"] = ~(df["arrival_is_canceled"] | df["departure_is_canceled"])
# is_canceled = the scheduled stop was not served and no replacement ran
df["is_canceled"] = ~df["ran"] & ~df["has_replacement"]

print(f"Loaded {len(df):,} records from {len(files)} files")
print(f"Files: {[f.name for f in files]}")

Loaded 29,976,896 records from 2 files
Files: ['data-2026-09.parquet', 'data-2026-10.parquet']


In [3]:
# Calculate statistics for all trains per station
all_stats = (
    df.groupby("station_name")
    .agg(
        average_delay=("delay_in_min", lambda x: x[df.loc[x.index, "ran"]].mean()),
        cancellation_rate=("is_canceled", "mean"),
        sample_size=("station_name", "size"),
    )
    .reset_index()
)
all_stats["train_type"] = "alle Züge"

# Calculate statistics by train type per station
type_stats = (
    df.groupby(["station_name", "train_type"])
    .agg(
        average_delay=("delay_in_min", lambda x: x[df.loc[x.index, "ran"]].mean()),
        cancellation_rate=("is_canceled", "mean"),
        sample_size=("station_name", "size"),
    )
    .reset_index()
)

# Combine all_stats and type_stats
combined_stats = pd.concat([all_stats, type_stats], ignore_index=True)

# Round the numeric columns
combined_stats["average_delay"] = combined_stats["average_delay"].round(2).fillna(0)
combined_stats["cancellation_rate"] = combined_stats["cancellation_rate"].round(2)

In [4]:
# Create a dictionary where each station has its table HTML
station_tables = {}
for station in sorted(combined_stats["station_name"].unique()):
    station_df = (
        combined_stats[combined_stats["station_name"] == station]
        .sort_values("sample_size", ascending=False)[
            ["train_type", "average_delay", "cancellation_rate", "sample_size"]
        ]
        .rename(
            columns={
                "train_type": "Zuggattung",
                "average_delay": "Durchschnittliche Verspätung [min]",
                "cancellation_rate": "Ausfallquote",
                "sample_size": "Anzahl Zughalte",
            }
        )
    )
    station_tables[station] = table(station_df)

# Set default station
default_station = "Düsseldorf Hbf" if "Düsseldorf Hbf" in station_tables else next(iter(station_tables.keys()))

content = cell(
    heading("Wie ist die durchschnittliche Verspätung von Zügen in einem Bahnhof?", level=1)
    + paragraph(
        "Hier kann man einen Bahnhof auswählen und die durchschnittliche Verspätung und Ausfallquote für die verschiedenen Zuggattungen sehen. "
        "Damit lässt sich auch sehen welche Zuggattungen wie oft an dem ausgewählten Bahnhof halten."
    )
    + dropdown(station_tables, default=default_station)
)
display(HTML(content))

In [5]:
start_date = files[0].stem.replace("data-", "")
end_date = files[-1].stem.replace("data-", "")
content = cell(
    f'<p style="font-size: 0.8em; text-align: center;">Quelle: <a href="https://github.com/piebro/deutsche-bahn-data/blob/main/notebooks/bahnhof.ipynb">Berechnet</a> auf Basis von <a href="https://github.com/piebro/deutsche-bahn-data">gesammelten Daten</a> von der Deutschen Bahn vom {start_date} bis {end_date}.</p>'
)
display(HTML(content))